<a href="https://colab.research.google.com/github/kunphat510214-netizen/project6/blob/Step-2/%E0%B8%82%E0%B8%B1%E0%B9%89%E0%B8%99%E0%B8%95%E0%B8%AD%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%882_%E0%B8%AD%E0%B8%AD%E0%B8%81%E0%B9%81%E0%B8%9A%E0%B8%9A_Class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import sqlite3
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print


class Member:
    def __init__(self, member_id, name, points=0):
        self.member_id = member_id
        self.name = name
        self.points = points

    def add_points(self, cups=1):
        self.points += cups
        return self.points


class DrinkOrder:
    BASE_PRICES = {
        "อเมริกาโน่": 45, "ลาเต้": 50, "คาปูชิโน่": 50,
        "เอสเพรสโซ่": 50, "มอคค่า": 50, "ชาเขียว": 45,
        "ชาไทย": 45, "มัทฉะลาเต้": 50, "ช็อกโกแลต": 50,
        "นมสด": 40, "สตรอว์เบอร์รี่โซดา": 40,
        "บลูฮาวายโซดา": 40, "ยูซุโซดา": 40,
    }
    TYPE_PRICES = {"ร้อน": 0, "เย็น": 5, "ปั่น": 10}
    SIZE_PRICES = {"S": 0, "M": 10, "L": 20}
    TOPPING_PRICES = {"ไม่มี": 0, "ไข่มุก": 5, "วิปครีม": 15, "ครีมชีส": 20, "เจลลี่": 5}
    PAYMENT_METHODS = ("เงินสด", "QR Code", "บัตร")

    def __init__(self, order_id, customer_name, menu_name, drink_type, size,
                 topping, sweetness_level, receive_type, member_id=None):
        self.order_id = order_id
        self.queue_no = f"Q{order_id:03d}"
        self.customer_name = customer_name
        self.menu_name = menu_name
        self.drink_type = drink_type
        self.size = size
        self.topping = topping
        self.sweetness_level = sweetness_level
        self.receive_type = receive_type
        self.member_id = member_id
        self.payment_method = None
        self.payment_status = "ยังไม่ชำระ"
        self.status = "รอชำระเงิน"
        self.wait_minutes = 0

    def calculate_subtotal(self):
        return (self.BASE_PRICES[self.menu_name]
                + self.TYPE_PRICES[self.drink_type]
                + self.SIZE_PRICES[self.size]
                + self.TOPPING_PRICES[self.topping])

    def calculate_price(self):
        # สมาชิกได้รับส่วนลด 5%
        discount_rate = 0.05 if self.member_id else 0
        return round(self.calculate_subtotal() * (1 - discount_rate), 2)

    def confirm_payment(self, payment_method):
        if payment_method not in self.PAYMENT_METHODS:
            raise ValueError("ช่องทางชำระเงินไม่ถูกต้อง")
        self.payment_method = payment_method
        self.payment_status = "ชำระแล้ว"
        self.status = "รอดำเนินการ"

    def receipt_text(self):
        return (f"ใบเสร็จ R{self.order_id:05d} | {self.queue_no} | "
                f"{self.menu_name} {self.drink_type} {self.size} | "
                f"{self.calculate_price():,.2f} บาท | {self.payment_method}")

    def sticker_text(self):
        return (f"{self.queue_no} {self.menu_name}/{self.drink_type}/{self.size} "
                f"หวาน {self.sweetness_level} ท็อปปิ้ง {self.topping} ({self.receive_type})")

    def to_record(self):
        return {
            "order_id": self.order_id,
            "queue_no": self.queue_no,
            "customer_name": self.customer_name,
            "member_id": self.member_id,
            "menu_name": self.menu_name,
            "drink_type": self.drink_type,
            "size": self.size,
            "topping": self.topping,
            "sweetness_level": self.sweetness_level,
            "receive_type": self.receive_type,
            "payment_method": self.payment_method,
            "subtotal": self.calculate_subtotal(),
            "price": self.calculate_price(),
            "wait_minutes": self.wait_minutes,
            "status": self.status,
        }


class QueueSystem:
    def __init__(self):
        self.active_orders = []
        self.events = []

    def _log(self, order, event):
        self.events.append({
            "event_id": len(self.events) + 1,
            "order_id": order.order_id,
            "event": event,
            "status": order.status,
        })

    def send_to_kds(self, order):
        if order.payment_status != "ชำระแล้ว":
            raise ValueError("ต้องยืนยันการชำระเงินก่อนส่งเข้า KDS")
        self.active_orders.append(order)
        order.status = "รอดำเนินการ"
        self._log(order, "ส่งเข้า KDS")

    def start_preparing(self, order):
        order.status = "กำลังทำ"
        self._log(order, "บาริสต้ารับออเดอร์")

    def call_queue(self, order, announce=False):
        order.status = "พร้อมรับ"
        self._log(order, "เรียกคิว")
        message = f"🔔 คิว {order.queue_no} พร้อมรับที่เคาน์เตอร์"
        if announce:
            print(message, "(เสียงเรียก)")
        return message

    def complete_order(self, order):
        order.status = "เสร็จสิ้น"
        self._log(order, "ส่งมอบสำเร็จ")
        self.active_orders = [o for o in self.active_orders if o.order_id != order.order_id]